# TRADES Paper Complete Reproduction

Reproduces **Table 1** and key figures from [TRADES: Generating Realistic Market Simulations with Diffusion Models](https://arxiv.org/abs/XXXX)

## Quick Start Guide:

1. **Set configuration** in Cell 2 (choose stocks, baselines, time window)
2. **Upload LOBSTER data** for the dates you need
3. **Run all cells** - the notebook handles optional baselines automatically
4. **View results** at the end - averaged predictive scores and plots

## What Gets Reproduced:

### Core (Always):
- **Table 1** values for selected stocks (averaged over 2 days)
- **Predictive Score**: TRADES vs Market Replay
- **Figure 2**: PCA distribution coverage
- **Figure 3**: Stylized facts validation

### Optional (if enabled):
- **Table 1** full comparison (TRADES vs CGAN)
- **Figure 3**: Multi-method stylized facts
- **Figure 5**: Volume distribution comparison

---

## Expected Results:

| Method        | Tesla | Intel |
|--------------|-------|-------|
| Market Replay| 0.923 | 0.149 |
| CGAN         | 3.453 | 0.699 |
| **TRADES**   | **1.213** | **0.307** |

*Note: Your results may vary slightly due to random seeds and different time windows*


In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================

# Stocks to simulate
RUN_INTC = True   # Intel (recommended - main paper focus)
RUN_TSLA = True  # Tesla (optional - full reproduction)

# Optional baselines (increase runtime significantly)
RUN_CGAN = True  # Wasserstein GAN (~7 hours total for 2 days)
RUN_FINETUNED_TRADES = True  # Finetuned TRADES with news features

# Simulation parameters
START_TIME = '10:00:00'
END_TIME = '12:00:00'  # 2-hour window (paper default)
# For faster testing: END_TIME = '10:30:00' (30-min window)

# Display configuration
stocks = []
if RUN_INTC: stocks.append('INTC')
if RUN_TSLA: stocks.append('TSLA')

baselines = ['TRADES', 'Market Replay']
if RUN_CGAN: baselines.append('CGAN')
if RUN_FINETUNED_TRADES: baselines.append('Finetuned TRADES')

print("="*60)
print("TRADES PAPER REPRODUCTION - CONFIGURATION")
print("="*60)
print(f"\nStocks: {', '.join(stocks) if stocks else 'None selected!'}")
print(f"Baselines: {', '.join(baselines)}")
print(f"Time window: {START_TIME} - {END_TIME}")

# Estimate runtime
window_hours = 2 if END_TIME == '12:00:00' else 0.5
trades_time = window_hours * 3.5  # TRADES takes ~3.5x real-time
replay_time = window_hours * 0.08  # Replay is fast
cgan_time = window_hours * 3.5 if RUN_CGAN else 0
finetuned_time = window_hours * 3.5 if RUN_FINETUNED_TRADES else 0

total_per_day = trades_time + replay_time + cgan_time + finetuned_time
total_time = total_per_day * 2 * len(stocks)  # 2 days per stock

print(f"\nEstimated runtime:")
print(f"  Per day per stock: ~{total_per_day:.1f} hours")
print(f"  Total (2 days x {len(stocks)} stock(s)): ~{total_time:.1f} hours")
print(f"\n  Breakdown:")
print(f"    - TRADES: ~{trades_time:.1f}h per day")
print(f"    - Market Replay: ~{replay_time*60:.0f} min per day")
if RUN_CGAN:
    print(f"    - CGAN: ~{cgan_time:.1f}h per day")
if RUN_FINETUNED_TRADES:
    print(f"    - Finetuned TRADES: ~{finetuned_time:.1f}h per day")
print("="*60)

if not stocks:
    print("\n⚠️  WARNING: No stocks selected! Set RUN_INTC=True or RUN_TSLA=True")

TRADES PAPER REPRODUCTION - CONFIGURATION

Stocks: INTC, TSLA
Baselines: TRADES, Market Replay, CGAN, Finetuned TRADES
Time window: 10:00:00 - 12:00:00

Estimated runtime:
  Per day per stock: ~21.2 hours
  Total (2 days x 2 stock(s)): ~84.6 hours

  Breakdown:
    - TRADES: ~7.0h per day
    - Market Replay: ~10 min per day
    - CGAN: ~7.0h per day
    - Finetuned TRADES: ~7.0h per day


---\n# Section 1: Environment Setup

## 1.1. Clone Repository

In [ ]:
# Clone D-MEADS repository
!git clone https://github.com/tiannanzhang/D-MEADS
%cd D-MEADS
!pwd

Cloning into 'D-MEADS'...
remote: Enumerating objects: 4398, done.
remote: Counting objects: 100% (151/151), done.
remote: Compressing objects: 100% (118/118), done.
remote: Total 4398 (delta 52), reused 108 (delta 30), pack-reused 4247 (from 2)
Receiving objects: 100% (4398/4398), 688.36 MiB | 15.09 MiB/s, done.
Resolving deltas: 100% (2465/2465), done.
Filtering content: 100% (5/5), 4.43 MiB | 4.07 MiB/s, done.
/content/D-MEADS
/content/D-MEADS


## 1.2. Install Dependencies

In [ ]:
!pip install -r requirements.txt

print("✅ Dependencies installed")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.9/44.9 kB 3.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.0/846.0 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.5/81.5 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 49.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.2/983.2 kB 66.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 849.5/849.5 kB 60.3 MB/s eta 0:00:00
  Created wheel for sgmllib3k: filename=sgmllib3k-1.0.0-py3-none-any.whl size=6046 sha256=f4ca85e6482f97d4de2019ae8519dfd561023e547aeeeb7fee5a92a9d23a78bc
  Stored in directory: /root/.cache/pip/wheels/03/f5/1a/23761066dac1d0e8e683e5fdb27e12de53209d05a4a37e6246
Successfully built sgmllib3k
✅ Dependencies installed


## 1.3. Check Resources

In [ ]:
import torch

print("GPU:", "Available" if torch.cuda.is_available() else "Not available")
if torch.cuda.is_available():
    print(f"  - {torch.cuda.get_device_name(0)}")
    print(f"  - {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

print("\nRAM:")
!free -h | grep Mem

print("\n⚠️  Requirements:")
print("  - 2-hour simulations: ~25GB RAM (Colab Pro recommended)")
print("  - 30-min simulations: ~12GB RAM (free Colab may work)")

GPU: Available
  - NVIDIA A100-SXM4-40GB
  - 42.5 GB

RAM:
Mem:            83Gi       1.4Gi        76Gi       2.0Mi       5.7Gi        81Gi

⚠️  Requirements:
  - 2-hour simulations: ~25GB RAM (Colab Pro recommended)
  - 30-min simulations: ~12GB RAM (free Colab may work)


## 1.4. Download Checkpoints

In [ ]:
import os

!mkdir -p data/checkpoints/TRADES
!mkdir -p data/checkpoints/CGAN
!mkdir -p data/checkpoints/NEWS_TRADES

# TRADES checkpoints (download from Google Drive)
print("Downloading TRADES checkpoints...")
checkpoint_url = "https://drive.google.com/drive/folders/1fg5G9KzmzC6E4FUYSCjObJ7sCEdjo43W"
!gdown --folder {checkpoint_url} -O data/checkpoints/TRADES/ --quiet

# Move checkpoints out of CDT subfolder if it exists
if os.path.exists("data/checkpoints/TRADES/CDT"):
    print("Moving TRADES checkpoints out of CDT folder...")
    !mv data/checkpoints/TRADES/CDT/*.ckpt data/checkpoints/TRADES/ 2>/dev/null || true
    !rm -rf data/checkpoints/TRADES/CDT
    print("✓ Checkpoints moved to data/checkpoints/TRADES/")

# Finetuned TRADES checkpoints with news features
print("\nDownloading Finetuned TRADES checkpoints...")
finetuned_checkpoint_url = "https://drive.google.com/drive/folders/1E3vHoSg0IopTx-jQ2SJj35h1rCB_amaK"
!gdown --folder {finetuned_checkpoint_url} -O data/checkpoints/NEWS_TRADES/ --quiet

if os.path.exists("data/checkpoints/NEWS_TRADES/NEWS_TRADES"):
    print("Moving NEWS_TRADES checkpoints out of NEWS_TRADES folder...")
    !mv data/checkpoints/NEWS_TRADES/NEWS_TRADES/*.ckpt data/checkpoints/NEWS_TRADES/ 2>/dev/null || true
    !rm -rf data/checkpoints/NEWS_TRADES/NEWS_TRADES
    print("✓ Checkpoints moved to data/checkpoints/NEWS_TRADES/NEWS_TRADES")
print("\nCheckpoint status:")
!ls -lh data/checkpoints/TRADES/*.ckpt 2>/dev/null | wc -l | xargs -I {} echo "  TRADES: {} checkpoint(s)"
!ls -lh data/checkpoints/NEWS_TRADES/*.ckpt 2>/dev/null | wc -l | xargs -I {} echo "  Finetuned TRADES: {} checkpoint(s)"
!ls -lh data/checkpoints/CGAN/*.ckpt 2>/dev/null | wc -l | xargs -I {} echo "  CGAN: {} checkpoint(s)"

print("\nIf download failed, manually download from:")
print("  TRADES: https://drive.google.com/drive/folders/1fg5G9KzmzC6E4FUYSCjObJ7sCEdjo43W")
print("  Finetuned TRADES: https://drive.google.com/drive/folders/1E3vHoSg0IopTx-jQ2SJj35h1rCB_amaK")

Moving TRADES checkpoints out of CDT folder...
✓ Checkpoints moved to data/checkpoints/TRADES/

Moving NEWS_TRADES checkpoints out of NEWS_TRADES folder...
✓ Checkpoints moved to data/checkpoints/NEWS_TRADES/NEWS_TRADES

Checkpoint status:
  TRADES: 2 checkpoint(s)
  Finetuned TRADES: 2 checkpoint(s)
  CGAN: 2 checkpoint(s)

If download failed, manually download from:
  TRADES: https://drive.google.com/drive/folders/1fg5G9KzmzC6E4FUYSCjObJ7sCEdjo43W
  Finetuned TRADES: https://drive.google.com/drive/folders/1E3vHoSg0IopTx-jQ2SJj35h1rCB_amaK


## 1.5. Update Configuration Files

Fix the DATE_TRADING_DAYS to match our simulation dates.

In [ ]:
# Update constants.py
with open('constants.py', 'r') as f:
    content = f.read()

content = content.replace(
    'DATE_TRADING_DAYS = ["2015-01-02", "2015-01-30"]',
    'DATE_TRADING_DAYS = ["2015-01-29", "2015-01-30"]'
)

with open('constants.py', 'w') as f:
    f.write(content)

print("✅ Configuration updated:")
print("   DATE_TRADING_DAYS = ['2015-01-29', '2015-01-30']")

✅ Configuration updated:
   DATE_TRADING_DAYS = ['2015-01-29', '2015-01-30']


---
# Section 2: Data Upload

**Required LOBSTER files for each stock:**

For each date (2015-01-29 and 2015-01-30), you need:
- `{STOCK}_2015-01-{DAY}_34200000_57600000_message_10.csv`
- `{STOCK}_2015-01-{DAY}_34200000_57600000_orderbook_10.csv`

Where:
- `34200000` = 09:30:00 (market open in seconds)
- `57600000` = 16:00:00 (market close)
- `10` = top 10 levels of the order book

**Total files needed:**
- INTC: 4 files (2 days × 2 file types)
- TSLA: 4 files (2 days × 2 file types)


## 2.1. Upload INTC Data (if selected)

In [ ]:
from google.colab import files
import os
import zipfile

if not RUN_INTC:
    print("⏭️  INTC not selected, skipping...")
else:
    data_dir = "data/INTC/INTC_2015-01-29_2015-01-30"
    os.makedirs(data_dir, exist_ok=True)

    print("Upload INTC data as a ZIP file")
    print("The ZIP should contain a folder that contains:")
    print("  - INTC_2015-01-29_34200000_57600000_message_10.csv")
    print("  - INTC_2015-01-29_34200000_57600000_orderbook_10.csv")
    print("  - INTC_2015-01-30_34200000_57600000_message_10.csv")
    print("  - INTC_2015-01-30_34200000_57600000_orderbook_10.csv")

    uploaded = files.upload()

    # Process uploaded zip files
    for filename in uploaded.keys():
        if filename.endswith('.zip'):
            print(f"\nExtracting {filename}...")
            with zipfile.ZipFile(filename, 'r') as zip_ref:
                zip_ref.extractall(data_dir)
            os.remove(filename)  # Clean up the zip file
            print(f"✓ Extracted to {data_dir}")
        else:
            # If individual CSV files are uploaded, move them
            !mv "{filename}" "{data_dir}/"
            print(f"✓ {filename}")

    print(f"\n✅ INTC data ready in {data_dir}")
    !ls -lh {data_dir}

Upload INTC data as a ZIP file
The ZIP should contain a folder that contains:
  - INTC_2015-01-29_34200000_57600000_message_10.csv
  - INTC_2015-01-29_34200000_57600000_orderbook_10.csv
  - INTC_2015-01-30_34200000_57600000_message_10.csv
  - INTC_2015-01-30_34200000_57600000_orderbook_10.csv


Saving INTC_2015-01-29_2015-01-30.zip to INTC_2015-01-29_2015-01-30.zip

Extracting INTC_2015-01-29_2015-01-30.zip...
✓ Extracted to data/INTC/INTC_2015-01-29_2015-01-30

✅ INTC data ready in data/INTC/INTC_2015-01-29_2015-01-30
total 527M
-rw-r--r-- 1 root root  41M Dec  6 19:08 INTC_2015-01-29_34200000_57600000_message_10.csv
-rw-r--r-- 1 root root 216M Dec  6 19:08 INTC_2015-01-29_34200000_57600000_orderbook_10.csv
-rw-r--r-- 1 root root  43M Dec  6 19:08 INTC_2015-01-30_34200000_57600000_message_10.csv
-rw-r--r-- 1 root root 227M Dec  6 19:08 INTC_2015-01-30_34200000_57600000_orderbook_10.csv


## 2.2. Upload TSLA Data (if selected)

In [ ]:
if not RUN_TSLA:
    print("⏭️  TSLA not selected, skipping...")
else:
    data_dir = "data/TSLA/TSLA_2015-01-29_2015-01-30"
    os.makedirs(data_dir, exist_ok=True)

    print("Upload TSLA data as a ZIP file")
    print("The ZIP should contain a folder that contains:")
    print("  - TSLA_2015-01-29_34200000_57600000_message_10.csv")
    print("  - TSLA_2015-01-29_34200000_57600000_orderbook_10.csv")
    print("  - TSLA_2015-01-30_34200000_57600000_message_10.csv")
    print("  - TSLA_2015-01-30_34200000_57600000_orderbook_10.csv")

    uploaded = files.upload()

    # Process uploaded zip files
    for filename in uploaded.keys():
        if filename.endswith('.zip'):
            print(f"\nExtracting {filename}...")
            with zipfile.ZipFile(filename, 'r') as zip_ref:
                zip_ref.extractall(data_dir)
            os.remove(filename)  # Clean up the zip file
            print(f"✓ Extracted to {data_dir}")
        else:
            # If individual CSV files are uploaded, move them
            !mv "{filename}" "{data_dir}/"
            print(f"✓ {filename}")

    print(f"\n✅ TSLA data ready in {data_dir}")
    !ls -lh {data_dir}

Upload TSLA data as a ZIP file
The ZIP should contain a folder that contains:
  - TSLA_2015-01-29_34200000_57600000_message_10.csv
  - TSLA_2015-01-29_34200000_57600000_orderbook_10.csv
  - TSLA_2015-01-30_34200000_57600000_message_10.csv
  - TSLA_2015-01-30_34200000_57600000_orderbook_10.csv


Saving TSLA_2015-01-29_2015-01-30.zip to TSLA_2015-01-29_2015-01-30.zip

Extracting TSLA_2015-01-29_2015-01-30.zip...
✓ Extracted to data/TSLA/TSLA_2015-01-29_2015-01-30

✅ TSLA data ready in data/TSLA/TSLA_2015-01-29_2015-01-30
total 168M
-rw-r--r-- 1 root root 14M Dec  6 19:10 TSLA_2015-01-29_34200000_57600000_message_10.csv
-rw-r--r-- 1 root root 69M Dec  6 19:10 TSLA_2015-01-29_34200000_57600000_orderbook_10.csv
-rw-r--r-- 1 root root 15M Dec  6 19:10 TSLA_2015-01-30_34200000_57600000_message_10.csv
-rw-r--r-- 1 root root 72M Dec  6 19:10 TSLA_2015-01-30_34200000_57600000_orderbook_10.csv


## 2.3. Clean Data Files

LOBSTER data sometimes contains a "null" column that causes errors. This cell removes it.

In [ ]:
import pandas as pd
import glob

def clean_lobster_data(stock):
    data_dir = f"data/{stock}/{stock}_2015-01-29_2015-01-30"
    csv_files = glob.glob(f"{data_dir}/*.csv")

    if not csv_files:
        print(f"⚠️  No files found for {stock}")
        return

    print(f"Cleaning {stock} data ({len(csv_files)} files)...")
    for csv_file in csv_files:
        df = pd.read_csv(csv_file, header=None, na_values=['null', 'NULL', 'Null'])
        null_ratio = df.iloc[:, -1].isnull().sum() / len(df)

        if null_ratio > 0.9:
            print(f"  ✓ {os.path.basename(csv_file)}: removed null column")
            df = df.iloc[:, :-1]
            df.to_csv(csv_file, header=False, index=False)
        else:
            print(f"  - {os.path.basename(csv_file)}: OK")

    print(f"✅ {stock} data cleaned\n")

if RUN_INTC:
    clean_lobster_data('INTC')
if RUN_TSLA:
    clean_lobster_data('TSLA')

print("All data ready for simulation!")

Cleaning INTC data (4 files)...
  - INTC_2015-01-30_34200000_57600000_orderbook_10.csv: OK
  ✓ INTC_2015-01-29_34200000_57600000_message_10.csv: removed null column
  - INTC_2015-01-29_34200000_57600000_orderbook_10.csv: OK
  ✓ INTC_2015-01-30_34200000_57600000_message_10.csv: removed null column
✅ INTC data cleaned

Cleaning TSLA data (4 files)...
  - TSLA_2015-01-29_34200000_57600000_orderbook_10.csv: OK
  ✓ TSLA_2015-01-30_34200000_57600000_message_10.csv: removed null column
  ✓ TSLA_2015-01-29_34200000_57600000_message_10.csv: removed null column
  - TSLA_2015-01-30_34200000_57600000_orderbook_10.csv: OK
✅ TSLA data cleaned

All data ready for simulation!


## 2.4. Upload News Features (for Finetuned TRADES)

If you plan to run Finetuned TRADES, you need to upload the news feature files.

In [ ]:
from google.colab import files
import pandas as pd
import os

if not RUN_TSLA or not RUN_FINETUNED_TRADES:
    print("⏭️  TSLA finetuned TRADES not selected, skipping...")
else:
    data_dir = "data/TSLA"
    os.makedirs(data_dir, exist_ok=True)

    print("Upload TSLA news CSV files")
    print("Expected files:")
    print("  1. test_news_2015-01-29.csv")
    print("  2. test_news_2015-01-30.csv")
    print("Each CSV should have columns: timestamp, sentiment, headline_count")

    uploaded = files.upload()

    # Expected files
    expected_files = ['test_news_2015-01-29.csv', 'test_news_2015-01-30.csv']
    found_files = []

    # Move uploaded files
    for filename in uploaded.keys():
        if filename in expected_files:
            !mv "{filename}" "{data_dir}/{filename}"
            found_files.append(filename)
            print(f"✓ {filename} uploaded to {data_dir}/")
        else:
            print(f"⚠️  Unexpected file: '{filename}'")
            !mv "{filename}" "{data_dir}/"

    # Verify files
    print(f"\n{'='*60}")
    all_present = True
    for expected_file in expected_files:
        file_path = f"{data_dir}/{expected_file}"
        if os.path.exists(file_path):
            try:
                df = pd.read_csv(file_path)
                required_cols = ['timestamp', 'sentiment', 'headline_count']

                if all(col in df.columns for col in required_cols):
                    print(f"✅ {expected_file}: {len(df):,} rows")
                    print(f"   Columns: {', '.join(df.columns.tolist())}")
                else:
                    print(f"⚠️  {expected_file}: Missing required columns!")
                    print(f"   Expected: {required_cols}")
                    print(f"   Found: {df.columns.tolist()}")
                    all_present = False
            except Exception as e:
                print(f"❌ {expected_file}: Error reading file - {e}")
                all_present = False
        else:
            print(f"❌ {expected_file}: File not found!")
            all_present = False

    if all_present:
        print(f"\n✅ All TSLA news CSV files ready!")
    else:
        print(f"\n⚠️  Some TSLA news files are missing or invalid!")

Upload TSLA news CSV files
Expected files:
  1. test_news_2015-01-29.csv
  2. test_news_2015-01-30.csv
Each CSV should have columns: timestamp, sentiment, headline_count


Saving test_news_2015-01-29.csv to test_news_2015-01-29.csv
Saving test_news_2015-01-30.csv to test_news_2015-01-30.csv
✓ test_news_2015-01-29.csv uploaded to data/TSLA/
✓ test_news_2015-01-30.csv uploaded to data/TSLA/

✅ test_news_2015-01-29.csv: 302,923 rows
   Columns: timestamp, sentiment, headline_count
✅ test_news_2015-01-30.csv: 315,544 rows
   Columns: timestamp, sentiment, headline_count

✅ All TSLA news CSV files ready!


In [ ]:
from google.colab import files
import pandas as pd
import os

if not RUN_INTC or not RUN_FINETUNED_TRADES:
    print("⏭️  INTC finetuned TRADES not selected, skipping...")
else:
    data_dir = "data/INTC"
    os.makedirs(data_dir, exist_ok=True)

    print("Upload INTC news CSV files")
    print("Expected files:")
    print("  1. test_news_2015-01-29.csv")
    print("  2. test_news_2015-01-30.csv")
    print("Each CSV should have columns: timestamp, sentiment, headline_count")

    uploaded = files.upload()

    # Expected files
    expected_files = ['test_news_2015-01-29.csv', 'test_news_2015-01-30.csv']
    found_files = []

    # Move uploaded files
    for filename in uploaded.keys():
        if filename in expected_files:
            !mv "{filename}" "{data_dir}/{filename}"
            found_files.append(filename)
            print(f"✓ {filename} uploaded to {data_dir}/")
        else:
            print(f"⚠️  Unexpected file: '{filename}'")
            !mv "{filename}" "{data_dir}/"

    # Verify files
    print(f"\n{'='*60}")
    all_present = True
    for expected_file in expected_files:
        file_path = f"{data_dir}/{expected_file}"
        if os.path.exists(file_path):
            try:
                df = pd.read_csv(file_path)
                required_cols = ['timestamp', 'sentiment', 'headline_count']

                if all(col in df.columns for col in required_cols):
                    print(f"✅ {expected_file}: {len(df):,} rows")
                    print(f"   Columns: {', '.join(df.columns.tolist())}")
                else:
                    print(f"⚠️  {expected_file}: Missing required columns!")
                    print(f"   Expected: {required_cols}")
                    print(f"   Found: {df.columns.tolist()}")
                    all_present = False
            except Exception as e:
                print(f"❌ {expected_file}: Error reading file - {e}")
                all_present = False
        else:
            print(f"❌ {expected_file}: File not found!")
            all_present = False

    if all_present:
        print(f"\n✅ All INTC news CSV files ready!")
    else:
        print(f"\n⚠️  Some INTC news files are missing or invalid!")

Upload INTC news CSV files
Expected files:
  1. test_news_2015-01-29.csv
  2. test_news_2015-01-30.csv
Each CSV should have columns: timestamp, sentiment, headline_count


Saving test_news_2015-01-29.csv to test_news_2015-01-29.csv
Saving test_news_2015-01-30.csv to test_news_2015-01-30.csv
✓ test_news_2015-01-29.csv uploaded to data/INTC/
✓ test_news_2015-01-30.csv uploaded to data/INTC/

✅ test_news_2015-01-29.csv: 912,675 rows
   Columns: timestamp, sentiment, headline_count
✅ test_news_2015-01-30.csv: 956,561 rows
   Columns: timestamp, sentiment, headline_count

✅ All INTC news CSV files ready!


---
# Section 3: Simulations for 2015-01-29

We'll run 4 types of simulations (2 required, 2 optional):
1. **TRADES** (required) - Diffusion-based generation
2. **Market Replay** (required) - Real historical data
3. **CGAN** (optional) - GAN-based generation


In [ ]:
import glob
import os
import time

def get_latest_log(pattern):
    """Get the most recent log directory matching pattern"""
    logs = sorted(glob.glob(pattern), key=os.path.getmtime, reverse=True)
    return logs[0] if logs else None

def run_simulation(stock, date, method, start_time, end_time):
    """
    Run a simulation and return the log directory.

    Args:
        stock: Stock ticker (e.g., 'INTC', 'TSLA')
        date: Date in format 'YYYY-MM-DD' (e.g., '2015-01-29')
        method: One of 'TRADES', 'Finetuned_TRADES', 'Replay', 'CGAN'
        start_time: Start time (e.g., '10:00:00')
        end_time: End time (e.g., '12:00:00')

    Returns:
        Path to log directory containing processed_orders.csv
    """
    date_str = date.replace('-', '')
    print(f"\n{'='*60}")
    print(f"Running {method} for {stock} on {date}")
    print(f"Time window: {start_time} - {end_time}")
    print(f"{'='*60}\n")

    start = time.time()

    if method == "TRADES":
        # TRADES: diffusion ON + TRADES model
        # Log dir: world_agent_{stock}_{date}_{time}_{seed}_DDIM_...
        cmd = f"python -u ABIDES/abides.py -c world_agent_sim -t {stock} -date {date} -d True -m TRADES -st '{start_time}' -et '{end_time}'"
        print(f"Command: {cmd}\n")
        !{cmd}
        log_dir = get_latest_log(f"ABIDES/log/world_agent_{stock}_*TRADES*") or get_latest_log(f"ABIDES/log/world_agent_{stock}_*")

    elif method == "Finetuned_TRADES":
        # Finetuned TRADES: add -ft True flag with 10 DDIM steps for better quality
        # Log dir: finetuned_world_agent_{stock}_{date}_{time}_{seed}_DDIM_...
        cmd = f"python -u ABIDES/abides.py -c world_agent_sim -t {stock} -date {date} -d True -m TRADES -ft True -nsteps 10 -st '{start_time}' -et '{end_time}'"
        print(f"Command: {cmd}\n")
        !{cmd}
        log_dir = get_latest_log(f"ABIDES/log/finetuned_world_agent_{stock}_*")

    elif method == "Replay":
        # Market Replay: diffusion OFF (omit -d flag!)
        # Log dir: market_replay_{stock}_{date}_{time}_{seed}
        # IMPORTANT: Do NOT use -d False (type=bool bug makes it True!)
        cmd = f"python -u ABIDES/abides.py -c world_agent_sim -t {stock} -date {date} -st '{start_time}' -et '{end_time}'"
        print(f"Command: {cmd}\n")
        !{cmd}
        log_dir = get_latest_log(f"ABIDES/log/market_replay_{stock}_*")

    elif method == "CGAN":
        # CGAN: diffusion ON + CGAN model
        # Log dir: world_agent_{stock}_{date}_{time}_{seed}_DDIM_..._CGAN_...
        cmd = f"python -u ABIDES/abides.py -c world_agent_sim -t {stock} -date {date} -d True -m CGAN -st '{start_time}' -et '{end_time}'"
        print(f"Command: {cmd}\n")
        !{cmd}
        log_dir = get_latest_log(f"ABIDES/log/world_agent_{stock}_*CGAN*") or get_latest_log(f"ABIDES/log/world_agent_{stock}_*")

    else:
        raise ValueError(f"Unknown method: {method}")

    elapsed = time.time() - start
    print(f"\n{'='*60}")
    print(f"✅ {method} simulation completed in {elapsed/60:.1f} minutes")
    print(f"📁 Log directory: {log_dir}")
    print(f"{'='*60}\n")

    if not log_dir or not os.path.exists(f"{log_dir}/processed_orders.csv"):
        print(f"⚠️  WARNING: processed_orders.csv not found in {log_dir}")
        print(f"   Check logs for errors.")

    return log_dir

print("✅ Simulation helper functions defined (with verified commands)")
print("\n📝 Command patterns:")
print("  TRADES: -c world_agent_sim -d True -m TRADES")
print("  Finetuned TRADES: -c world_agent_sim -d True -m TRADES -ft True -nsteps 10")
print("  Replay: -c world_agent_sim (omit -d flag)")
print("  CGAN:   -c world_agent_sim -d True -m CGAN")

✅ Simulation helper functions defined (with verified commands)

📝 Command patterns:
  TRADES: -c world_agent_sim -d True -m TRADES
  Finetuned TRADES: -c world_agent_sim -d True -m TRADES -ft True -nsteps 10
  Replay: -c world_agent_sim (omit -d flag)
  CGAN:   -c world_agent_sim -d True -m CGAN


## 3.1. INTC Simulations - Day 1/29

In [ ]:
if not RUN_INTC:
    print("⏭️  Skipping INTC")
    INTC_DAY1_RESULTS = None
else:
    INTC_DAY1_RESULTS = {}

    # # TRADES (required)
    # log_dir = run_simulation('INTC', '2015-01-29', 'TRADES', START_TIME, END_TIME)
    # INTC_DAY1_RESULTS['TRADES'] = log_dir
    # print(f"✅ TRADES: {log_dir}")

    # # Market Replay (required)
    # log_dir = run_simulation('INTC', '2015-01-29', 'Replay', START_TIME, END_TIME)
    # INTC_DAY1_RESULTS['Replay'] = log_dir
    # print(f"✅ Replay: {log_dir}")

    # Finetuned TRADES (optional)
    if RUN_FINETUNED_TRADES:
        log_dir = run_simulation('INTC', '2015-01-29', 'Finetuned_TRADES', START_TIME, END_TIME)
        INTC_DAY1_RESULTS['Finetuned_TRADES'] = log_dir
        print(f"✅ Finetuned TRADES: {log_dir}")

# CGAN (optional)
# if RUN_CGAN:
#     log_dir = run_simulation('INTC', '2015-01-29', 'CGAN', START_TIME, END_TIME)
#     INTC_DAY1_RESULTS['CGAN'] = log_dir
#     print(f"✅ CGAN: {log_dir}")
# else:
#     print("⏭️  CGAN skipped (not enabled)")

print(f"\n{'='*60}")
print("INTC Day 1/29 Complete!")
print(f"{'='*60}")


Running Finetuned_TRADES for INTC on 2015-01-29
Time window: 10:00:00 - 12:00:00

Command: python -u ABIDES/abides.py -c world_agent_sim -t INTC -date 2015-01-29 -d True -m TRADES -ft True -nsteps 10 -st '10:00:00' -et '12:00:00'

ABIDES: Agent-Based Interactive Discrete Event Simulation

/content/D-MEADS
Simulation Start Time: 2025-12-06 03:53:32.069688
Configuration seed: 30

checkpoint used:  data/checkpoints/NEWS_TRADES/val_ema=2.269_epoch=3_INTC_finetuned_model.ckpt
News features enabled for finetuned model
Loading per-day news file: data/INTC/test_news_2015-01-29.csv
Filtered news from 912675 to 908085 rows (time window: 0s - 23343s from market open)
msg_recipient:  0 msg_type:  MessageType.WAKEUP msg:  None

--- Simulation time: 2015-01-29 10:00:00, messages processed: 0, wallclock elapsed: 0 days 00:00:00.000152 ---

Current time: 2015-01-29 10:00:00
Current time: 2015-01-29 10:00:00.000477
Current time: 2015-01-29 10:00:00.000484
Current time: 2015-01-29 10:00:00.006081
Curre

## 3.2. TSLA Simulations - Day 1/29 (Optional)

In [ ]:
if not RUN_TSLA:
    print("⏭️  Skipping TSLA")
    TSLA_DAY1_RESULTS = None
else:
    TSLA_DAY1_RESULTS = {}

    # log_dir = run_simulation('TSLA', '2015-01-29', 'TRADES', START_TIME, END_TIME)
    # TSLA_DAY1_RESULTS['TRADES'] = log_dir

    # log_dir = run_simulation('TSLA', '2015-01-29', 'Replay', START_TIME, END_TIME)
    # TSLA_DAY1_RESULTS['Replay'] = log_dir

    if RUN_FINETUNED_TRADES:
        log_dir = run_simulation('TSLA', '2015-01-29', 'Finetuned_TRADES', START_TIME, END_TIME)
        TSLA_DAY1_RESULTS['Finetuned_TRADES'] = log_dir

# if RUN_CGAN:
#     log_dir = run_simulation('TSLA', '2015-01-29', 'CGAN', START_TIME, END_TIME)
#     TSLA_DAY1_RESULTS['CGAN'] = log_dir

print("\nTSLA Day 1/29 Complete!")


Running Finetuned_TRADES for TSLA on 2015-01-29
Time window: 10:00:00 - 12:00:00

Command: python -u ABIDES/abides.py -c world_agent_sim -t TSLA -date 2015-01-29 -d True -m TRADES -ft True -nsteps 10 -st '10:00:00' -et '12:00:00'

ABIDES: Agent-Based Interactive Discrete Event Simulation

/content/D-MEADS/ABIDES/util/plotting/liquidity_telemetry.py:185: SyntaxWarning: invalid escape sequence '\ '
  axes[2].set_ylabel("$\\frac{\\mathrm{best\ ask\ size}}{\\mathrm{best\ ask\ size} + \\mathrm{best\ bid\ size}}$",
/content/D-MEADS
Simulation Start Time: 2025-12-06 01:48:56.115995
Configuration seed: 30

checkpoint used:  data/checkpoints/NEWS_TRADES/val_ema=0.8624_epoch=5_TSLA_finetuned_model.ckpt
News features enabled for finetuned model
Loading per-day news file: data/TSLA/test_news_2015-01-29.csv
Filtered news from 302923 to 295438 rows (time window: 0s - 22777s from market open)
msg_recipient:  0 msg_type:  MessageType.WAKEUP msg:  None

--- Simulation time: 2015-01-29 10:00:00, messag

---
# Section 4: Simulations for 2015-01-30

Same process for the second day.

## 4.1. INTC Simulations - Day 1/30

In [ ]:
if not RUN_INTC:
    print("⏭️  Skipping INTC")
    INTC_DAY2_RESULTS = None
else:
    INTC_DAY2_RESULTS = {}

    # log_dir = run_simulation('INTC', '2015-01-30', 'TRADES', START_TIME, END_TIME)
    # INTC_DAY2_RESULTS['TRADES'] = log_dir

    # log_dir = run_simulation('INTC', '2015-01-30', 'Replay', START_TIME, END_TIME)
    # INTC_DAY2_RESULTS['Replay'] = log_dir

    if RUN_FINETUNED_TRADES:
        log_dir = run_simulation('INTC', '2015-01-30', 'Finetuned_TRADES', START_TIME, END_TIME)
        INTC_DAY2_RESULTS['Finetuned_TRADES'] = log_dir

# if RUN_CGAN:
#     log_dir = run_simulation('INTC', '2015-01-30', 'CGAN', START_TIME, END_TIME)
#     INTC_DAY2_RESULTS['CGAN'] = log_dir

print("INTC Day 1/30 Complete!")


Running Finetuned_TRADES for INTC on 2015-01-30
Time window: 10:00:00 - 12:00:00

Command: python -u ABIDES/abides.py -c world_agent_sim -t INTC -date 2015-01-30 -d True -m TRADES -ft True -nsteps 10 -st '10:00:00' -et '12:00:00'

ABIDES: Agent-Based Interactive Discrete Event Simulation

/content/D-MEADS/ABIDES/util/plotting/liquidity_telemetry.py:185: SyntaxWarning: invalid escape sequence '\ '
  axes[2].set_ylabel("$\\frac{\\mathrm{best\ ask\ size}}{\\mathrm{best\ ask\ size} + \\mathrm{best\ bid\ size}}$",
/content/D-MEADS
Simulation Start Time: 2025-12-06 19:26:39.862137
Configuration seed: 30

checkpoint used:  data/checkpoints/NEWS_TRADES/val_ema=2.269_epoch=3_INTC_finetuned_model.ckpt
News features enabled for finetuned model
Loading per-day news file: data/INTC/test_news_2015-01-30.csv
Filtered news from 956561 to 951588 rows (time window: 0s - 23351s from market open)
msg_recipient:  0 msg_type:  MessageType.WAKEUP msg:  None

--- Simulation time: 2015-01-30 10:00:00, message

## 4.2. TSLA Simulations - Day 1/30 (Optional)

In [ ]:
if not RUN_TSLA:
    print("⏭️  Skipping TSLA")
    TSLA_DAY2_RESULTS = None
else:
    TSLA_DAY2_RESULTS = {}

    # log_dir = run_simulation('TSLA', '2015-01-30', 'TRADES', START_TIME, END_TIME)
    # TSLA_DAY2_RESULTS['TRADES'] = log_dir

    # log_dir = run_simulation('TSLA', '2015-01-30', 'Replay', START_TIME, END_TIME)
    # TSLA_DAY2_RESULTS['Replay'] = log_dir

    if RUN_FINETUNED_TRADES:
        log_dir = run_simulation('TSLA', '2015-01-30', 'Finetuned_TRADES', START_TIME, END_TIME)
        TSLA_DAY2_RESULTS['Finetuned_TRADES'] = log_dir

# if RUN_CGAN:
#     log_dir = run_simulation('TSLA', '2015-01-30', 'CGAN', START_TIME, END_TIME)
#     TSLA_DAY2_RESULTS['CGAN'] = log_dir

print("TSLA Day 1/30 Complete!")


Running Finetuned_TRADES for TSLA on 2015-01-30
Time window: 10:00:00 - 12:00:00

Command: python -u ABIDES/abides.py -c world_agent_sim -t TSLA -date 2015-01-30 -d True -m TRADES -ft True -nsteps 10 -st '10:00:00' -et '12:00:00'

ABIDES: Agent-Based Interactive Discrete Event Simulation

/content/D-MEADS
Simulation Start Time: 2025-12-06 02:44:43.024654
Configuration seed: 30

checkpoint used:  data/checkpoints/NEWS_TRADES/val_ema=0.8624_epoch=5_TSLA_finetuned_model.ckpt
News features enabled for finetuned model
Loading per-day news file: data/TSLA/test_news_2015-01-30.csv
Filtered news from 315544 to 313109 rows (time window: 0s - 23123s from market open)
msg_recipient:  0 msg_type:  MessageType.WAKEUP msg:  None

--- Simulation time: 2015-01-30 10:00:00, messages processed: 0, wallclock elapsed: 0 days 00:00:00.000129 ---

Current time: 2015-01-30 10:00:00
Current time: 2015-01-30 10:00:00.150459
Current time: 2015-01-30 10:00:00.150460
Current time: 2015-01-30 10:00:00.150461
Curr

In [ ]:
# INTC_DAY1_RESULTS = {}
INTC_DAY1_RESULTS['TRADES'] = "/content/D-MEADS/ABIDES/log/world_agent_INTC_2015-01-29_12-00-00_30_DDIM_0.0_10_val_ema=2.317"
INTC_DAY1_RESULTS['Replay'] = "/content/D-MEADS/ABIDES/log/market_replay_INTC_2015-01-29_12-00-00_30"
INTC_DAY1_RESULTS['CGAN'] = "/content/D-MEADS/ABIDES/log/world_agent_INTC_2015-01-29_12-00-00_30_DDIM_0.0_1_val_ema=-1.05"
# INTC_DAY1_RESULTS['Finetuned_TRADES'] = "/content/D-MEADS/ABIDES/log/finetuned_world_agent_INTC_2015-01-29_12-00-00_30_DDIM_0.0_10_val_ema=2.269"
# TSLA_DAY1_RESULTS = {}
TSLA_DAY1_RESULTS['TRADES'] = "/content/D-MEADS/ABIDES/log/world_agent_TSLA_2015-01-29_12-00-00_30_DDIM_0.0_10_val_ema=0.811"
TSLA_DAY1_RESULTS['Replay'] = "/content/D-MEADS/ABIDES/log/market_replay_TSLA_2015-01-29_12-00-00_30"
TSLA_DAY1_RESULTS['CGAN'] = "/content/D-MEADS/ABIDES/log/world_agent_TSLA_2015-01-29_12-00-00_30_DDIM_0.0_1_val_ema=-1.04"
# TSLA_DAY1_RESULTS['Finetuned_TRADES'] = "/content/D-MEADS/ABIDES/log/finetuned_world_agent_TSLA_2015-01-29_12-00-00_30_DDIM_0.0_10_val_ema=0.862"

In [ ]:
# INTC_DAY2_RESULTS = {}
INTC_DAY2_RESULTS['TRADES'] = "/content/D-MEADS/ABIDES/log/world_agent_INTC_2015-01-30_12-00-00_30_DDIM_0.0_10_val_ema=2.317"
INTC_DAY2_RESULTS['Replay'] = "/content/D-MEADS/ABIDES/log/market_replay_INTC_2015-01-30_12-00-00_30"
INTC_DAY2_RESULTS['CGAN'] = "/content/D-MEADS/ABIDES/log/world_agent_INTC_2015-01-30_12-00-00_30_DDIM_0.0_1_val_ema=-1.05"
# INTC_DAY2_RESULTS['Finetuned_TRADES'] = "/content/D-MEADS/ABIDES/log/finetuned_world_agent_INTC_2015-01-30_12-00-00_30_DDIM_0.0_10_val_ema=2.269"
# TSLA_DAY2_RESULTS = {}
TSLA_DAY2_RESULTS['TRADES'] = "/content/D-MEADS/ABIDES/log/world_agent_TSLA_2015-01-30_12-00-00_30_DDIM_0.0_10_val_ema=0.811"
TSLA_DAY2_RESULTS['Replay'] = "/content/D-MEADS/ABIDES/log/market_replay_TSLA_2015-01-30_12-00-00_30"
TSLA_DAY2_RESULTS['CGAN'] = "/content/D-MEADS/ABIDES/log/world_agent_TSLA_2015-01-30_12-00-00_30_DDIM_0.0_1_val_ema=-1.04"
# TSLA_DAY2_RESULTS['Finetuned_TRADES'] = "/content/D-MEADS/ABIDES/log/finetuned_world_agent_TSLA_2015-01-30_12-00-00_30_DDIM_0.0_10_val_ema=0.862"

---
# Section 5: Evaluation

Calculate predictive scores for each day, then average them.

## 5.1. Compute Predictive Scores

In [ ]:
import sys
sys.path.append('evaluation/quantitative_eval')
import predictive_lstm

def compute_predictive_score(real_path, generated_path, label=""):
    """Compute predictive score for a single day"""
    print(f"\n{'='*60}")
    print(f"Computing Predictive Score: {label}")
    print(f"{'='*60}")
    print(f"Real data: {real_path}")
    print(f"Generated: {generated_path}")

    # Redirect output to capture MSE
    from io import StringIO
    old_stdout = sys.stdout
    sys.stdout = captured = StringIO()

    predictive_lstm.main(real_path, generated_path)

    sys.stdout = old_stdout
    output = captured.getvalue()

    # Extract MSE from output
    for line in output.split('\n'):
        if 'Test MSE:' in line:
            mse = float(line.split(':')[1].strip())
            print(f"  → MSE: {mse:.4f}")
            return mse

    print("  ⚠️  Could not extract MSE!")
    return None

# Storage for all scores
ALL_SCORES = {}

print("\n" + "="*80)
print("PREDICTIVE SCORE EVALUATION")
print("="*80)



PREDICTIVE SCORE EVALUATION


## 5.2. INTC Predictive Scores

In [ ]:
if not RUN_INTC:
    print("⏭️  INTC not run")
else:
    ALL_SCORES['INTC'] = {'day1': {}, 'day2': {}}

    # Day 1
    if INTC_DAY1_RESULTS:
        real_path = f"{INTC_DAY1_RESULTS['Replay']}/processed_orders.csv"
        trades_path = f"{INTC_DAY1_RESULTS['TRADES']}/processed_orders.csv"
        mse = compute_predictive_score(real_path, trades_path, "INTC Day 1/29 - TRADES")
        ALL_SCORES['INTC']['day1']['TRADES'] = mse

        if RUN_FINETUNED_TRADES and 'Finetuned_TRADES' in INTC_DAY1_RESULTS:
            ft_trades_path = f"{INTC_DAY1_RESULTS['Finetuned_TRADES']}/processed_orders.csv"
            mse = compute_predictive_score(real_path, ft_trades_path, "INTC Day 1/29 - Finetuned TRADES")
            ALL_SCORES['INTC']['day1']['Finetuned_TRADES'] = mse

        if RUN_CGAN and 'CGAN' in INTC_DAY1_RESULTS:
            cgan_path = f"{INTC_DAY1_RESULTS['CGAN']}/processed_orders.csv"
            mse = compute_predictive_score(real_path, cgan_path, "INTC Day 1/29 - CGAN")
            ALL_SCORES['INTC']['day1']['CGAN'] = mse

        # Market Replay score (baseline)
        mse = compute_predictive_score(real_path, real_path, "INTC Day 1/29 - Market Replay")
        ALL_SCORES['INTC']['day1']['Replay'] = mse

    # Day 2
    if INTC_DAY2_RESULTS:
        real_path = f"{INTC_DAY2_RESULTS['Replay']}/processed_orders.csv"
        trades_path = f"{INTC_DAY2_RESULTS['TRADES']}/processed_orders.csv"
        mse = compute_predictive_score(real_path, trades_path, "INTC Day 1/30 - TRADES")
        ALL_SCORES['INTC']['day2']['TRADES'] = mse

        if RUN_FINETUNED_TRADES and 'Finetuned_TRADES' in INTC_DAY2_RESULTS:
            ft_trades_path = f"{INTC_DAY2_RESULTS['Finetuned_TRADES']}/processed_orders.csv"
            mse = compute_predictive_score(real_path, ft_trades_path, "INTC Day 1/30 - Finetuned TRADES")
            ALL_SCORES['INTC']['day2']['Finetuned_TRADES'] = mse

        if RUN_CGAN and 'CGAN' in INTC_DAY2_RESULTS:
            cgan_path = f"{INTC_DAY2_RESULTS['CGAN']}/processed_orders.csv"
            mse = compute_predictive_score(real_path, cgan_path, "INTC Day 1/30 - CGAN")
            ALL_SCORES['INTC']['day2']['CGAN'] = mse

        mse = compute_predictive_score(real_path, real_path, "INTC Day 1/30 - Market Replay")
        ALL_SCORES['INTC']['day2']['Replay'] = mse

    print("\nINTC scores collected!")


Computing Predictive Score: INTC Day 1/29 - TRADES
Real data: /content/D-MEADS/ABIDES/log/market_replay_INTC_2015-01-29_12-00-00_30/processed_orders.csv
Generated: /content/D-MEADS/ABIDES/log/world_agent_INTC_2015-01-29_12-00-00_30_DDIM_0.0_10_val_ema=2.317/processed_orders.csv


  2%|▏         | 2/100 [00:12<10:27,  6.40s/it]


  → MSE: 0.1603

Computing Predictive Score: INTC Day 1/29 - Finetuned TRADES
Real data: /content/D-MEADS/ABIDES/log/market_replay_INTC_2015-01-29_12-00-00_30/processed_orders.csv
Generated: /content/D-MEADS/ABIDES/log/finetuned_world_agent_INTC_2015-01-29_12-00-00_30_DDIM_0.0_10_val_ema=2.269/processed_orders.csv


  2%|▏         | 2/100 [00:12<10:06,  6.19s/it]


  → MSE: 0.2234

Computing Predictive Score: INTC Day 1/29 - CGAN
Real data: /content/D-MEADS/ABIDES/log/market_replay_INTC_2015-01-29_12-00-00_30/processed_orders.csv
Generated: /content/D-MEADS/ABIDES/log/world_agent_INTC_2015-01-29_12-00-00_30_DDIM_0.0_1_val_ema=-1.05/processed_orders.csv


...

  → MSE: 0.3261

Computing Predictive Score: INTC Day 1/30 - CGAN
Real data: /content/D-MEADS/ABIDES/log/market_replay_INTC_2015-01-30_12-00-00_30/processed_orders.csv
Generated: /content/D-MEADS/ABIDES/log/world_agent_INTC_2015-01-30_12-00-00_30_DDIM_0.0_1_val_ema=-1.05/processed_orders.csv


  2%|▏         | 2/100 [00:10<08:25,  5.16s/it]


  → MSE: 0.1533

Computing Predictive Score: INTC Day 1/30 - Market Replay
Real data: /content/D-MEADS/ABIDES/log/market_replay_INTC_2015-01-30_12-00-00_30/processed_orders.csv
Generated: /content/D-MEADS/ABIDES/log/market_replay_INTC_2015-01-30_12-00-00_30/processed_orders.csv


  2%|▏         | 2/100 [00:56<46:24, 28.42s/it]


  → MSE: 0.1372

INTC scores collected!


## 5.3. TSLA Predictive Scores (Optional)

In [ ]:
if not RUN_TSLA:
    print("⏭️  TSLA not run")
else:
    ALL_SCORES['TSLA'] = {'day1': {}, 'day2': {}}

    # Day 1
    if TSLA_DAY1_RESULTS:
        real_path = f"{TSLA_DAY1_RESULTS['Replay']}/processed_orders.csv"
        trades_path = f"{TSLA_DAY1_RESULTS['TRADES']}/processed_orders.csv"
        mse = compute_predictive_score(real_path, trades_path, "TSLA Day 1/29 - TRADES")
        ALL_SCORES['TSLA']['day1']['TRADES'] = mse

        if RUN_FINETUNED_TRADES and 'Finetuned_TRADES' in TSLA_DAY1_RESULTS:
            ft_trades_path = f"{TSLA_DAY1_RESULTS['Finetuned_TRADES']}/processed_orders.csv"
            mse = compute_predictive_score(real_path, ft_trades_path, "TSLA Day 1/29 - Finetuned TRADES")
            ALL_SCORES['TSLA']['day1']['Finetuned_TRADES'] = mse

        if RUN_CGAN and 'CGAN' in TSLA_DAY1_RESULTS:
            cgan_path = f"{TSLA_DAY1_RESULTS['CGAN']}/processed_orders.csv"
            mse = compute_predictive_score(real_path, cgan_path, "TSLA Day 1/29 - CGAN")
            ALL_SCORES['TSLA']['day1']['CGAN'] = mse

        mse = compute_predictive_score(real_path, real_path, "TSLA Day 1/29 - Market Replay")
        ALL_SCORES['TSLA']['day1']['Replay'] = mse

    # Day 2
    if TSLA_DAY2_RESULTS:
        real_path = f"{TSLA_DAY2_RESULTS['Replay']}/processed_orders.csv"
        trades_path = f"{TSLA_DAY2_RESULTS['TRADES']}/processed_orders.csv"
        mse = compute_predictive_score(real_path, trades_path, "TSLA Day 1/30 - TRADES")
        ALL_SCORES['TSLA']['day2']['TRADES'] = mse

        if RUN_FINETUNED_TRADES and 'Finetuned_TRADES' in TSLA_DAY2_RESULTS:
            ft_trades_path = f"{TSLA_DAY2_RESULTS['Finetuned_TRADES']}/processed_orders.csv"
            mse = compute_predictive_score(real_path, ft_trades_path, "TSLA Day 1/30 - Finetuned TRADES")
            ALL_SCORES['TSLA']['day2']['Finetuned_TRADES'] = mse

        if RUN_CGAN and 'CGAN' in TSLA_DAY2_RESULTS:
            cgan_path = f"{TSLA_DAY2_RESULTS['CGAN']}/processed_orders.csv"
            mse = compute_predictive_score(real_path, cgan_path, "TSLA Day 1/30 - CGAN")
            ALL_SCORES['TSLA']['day2']['CGAN'] = mse

        mse = compute_predictive_score(real_path, real_path, "TSLA Day 1/30 - Market Replay")
        ALL_SCORES['TSLA']['day2']['Replay'] = mse

    print("\nTSLA scores collected!")


Computing Predictive Score: TSLA Day 1/29 - TRADES
Real data: /content/D-MEADS/ABIDES/log/market_replay_TSLA_2015-01-29_12-00-00_30/processed_orders.csv
Generated: /content/D-MEADS/ABIDES/log/world_agent_TSLA_2015-01-29_12-00-00_30_DDIM_0.0_10_val_ema=0.811/processed_orders.csv


 11%|█         | 11/100 [00:14<01:58,  1.33s/it]


  → MSE: 1.9312

Computing Predictive Score: TSLA Day 1/29 - Finetuned TRADES
Real data: /content/D-MEADS/ABIDES/log/market_replay_TSLA_2015-01-29_12-00-00_30/processed_orders.csv
Generated: /content/D-MEADS/ABIDES/log/finetuned_world_agent_TSLA_2015-01-29_12-00-00_30_DDIM_0.0_10_val_ema=0.862/processed_orders.csv


 10%|█         | 10/100 [00:15<02:22,  1.58s/it]


  → MSE: 1.9972

Computing Predictive Score: TSLA Day 1/29 - CGAN
Real data: /content/D-MEADS/ABIDES/log/market_replay_TSLA_2015-01-29_12-00-00_30/processed_orders.csv
Generated: /content/D-MEADS/ABIDES/log/world_agent_TSLA_2015-01-29_12-00-00_30_DDIM_0.0_1_val_ema=-1.04/processed_orders.csv


...

  → MSE: 1.0242

Computing Predictive Score: TSLA Day 1/30 - CGAN
Real data: /content/D-MEADS/ABIDES/log/market_replay_TSLA_2015-01-30_12-00-00_30/processed_orders.csv
Generated: /content/D-MEADS/ABIDES/log/world_agent_TSLA_2015-01-30_12-00-00_30_DDIM_0.0_1_val_ema=-1.04/processed_orders.csv


 10%|█         | 10/100 [00:16<02:30,  1.67s/it]


  → MSE: 1.5808

Computing Predictive Score: TSLA Day 1/30 - Market Replay
Real data: /content/D-MEADS/ABIDES/log/market_replay_TSLA_2015-01-30_12-00-00_30/processed_orders.csv
Generated: /content/D-MEADS/ABIDES/log/market_replay_TSLA_2015-01-30_12-00-00_30/processed_orders.csv


  4%|▍         | 4/100 [00:26<10:33,  6.60s/it]


  → MSE: 0.5449

TSLA scores collected!


---
# Section 6: Final Results

Compute averaged scores and compare to paper.

In [ ]:
import pandas as pd

print("\n" + "="*80)
print("FINAL RESULTS: Predictive Score (MAE) Averaged Over 2 Days")
print("="*80)

# Compute averages
results_table = []

for stock in ALL_SCORES:
    for method in ['Replay', 'TRADES', 'Finetuned_TRADES', 'CGAN']:
        day1_score = ALL_SCORES[stock]['day1'].get(method)
        day2_score = ALL_SCORES[stock]['day2'].get(method)

        if day1_score is not None and day2_score is not None:
            avg_score = (day1_score + day2_score) / 2
            results_table.append({
                'Stock': stock,
                'Method': method,
                'Day 1/29': f"{day1_score:.3f}",
                'Day 1/30': f"{day2_score:.3f}",
                'Average': f"{avg_score:.3f}"
            })

df_results = pd.DataFrame(results_table)

print("\nYour Results:")
print(df_results.to_string(index=False))

print("\n" + "="*80)
print("Paper Results (Table 1):")
print("="*80)
paper_results = """
Method         Tesla    Intel
Market Replay  0.923    0.149
CGAN           3.453    0.699
TRADES         1.213    0.307
"""
print(paper_results)

print("="*80)
print("\nInterpretation:")
print("  - Lower is better (closer to Market Replay)")
print("  - TRADES should significantly outperform CGAN")
print("  - Finetuned TRADES should outperform naive TRADES (if enabled)")
print("  - Your results may vary due to different time windows or random seeds")
print("="*80)


FINAL RESULTS: Predictive Score (MAE) Averaged Over 2 Days

Your Results:
Stock           Method Day 1/29 Day 1/30 Average
 INTC           Replay    0.132    0.137   0.135
 INTC           TRADES    0.160    0.213   0.187
 INTC Finetuned_TRADES    0.223    0.326   0.275
 INTC             CGAN    0.169    0.153   0.161
 TSLA           Replay    0.855    0.545   0.700
 TSLA           TRADES    1.931    1.320   1.626
 TSLA Finetuned_TRADES    1.997    1.024   1.511
 TSLA             CGAN    2.901    1.581   2.241

Paper Results (Table 1):

Method         Tesla    Intel
Market Replay  0.923    0.149
CGAN           3.453    0.699
TRADES         1.213    0.307


Interpretation:
  - Lower is better (closer to Market Replay)
  - TRADES should significantly outperform CGAN
  - Finetuned TRADES should outperform naive TRADES (if enabled)
  - Your results may vary due to different time windows or random seeds


## 6.1. Optional: Full main.py Evaluation

Run this if you want all the paper figures (PCA, stylized facts, etc.)

⚠️  This only works if you ran CGAN!

In [ ]:
RUN_FULL_EVAL = True  # Set to True to run full evaluation

if RUN_FULL_EVAL and not RUN_CGAN:
    print("⚠️  Cannot run full evaluation without CGAN!")
    print("   Set RUN_CGAN=True in configuration")
elif RUN_FULL_EVAL:
    print("Running full main.py evaluation...")

    import main

    # Run evaluation for all stock and day combinations
    evaluations = []

    if RUN_INTC and INTC_DAY1_RESULTS:
        evaluations.append(('INTC', 'Day 1 (2015-01-29)', INTC_DAY1_RESULTS))

    if RUN_INTC and INTC_DAY2_RESULTS:
        evaluations.append(('INTC', 'Day 2 (2015-01-30)', INTC_DAY2_RESULTS))

    if RUN_TSLA and TSLA_DAY1_RESULTS:
        evaluations.append(('TSLA', 'Day 1 (2015-01-29)', TSLA_DAY1_RESULTS))

    if RUN_TSLA and TSLA_DAY2_RESULTS:
        evaluations.append(('TSLA', 'Day 2 (2015-01-30)', TSLA_DAY2_RESULTS))

    for stock, day_label, results in evaluations:
        print(f"\n{'='*60}")
        print(f"Generating plots for {stock} {day_label}")
        print(f"{'='*60}")

        real_path = f"{results['Replay']}/processed_orders.csv"
        trades_path = f"{results['TRADES']}/processed_orders.csv"
        cgan_path = f"{results['CGAN']}/processed_orders.csv" if 'CGAN' in results else trades_path

        # Include finetuned TRADES if available
        if RUN_FINETUNED_TRADES and 'Finetuned_TRADES' in results:
            finetuned_trades_path = f"{results['Finetuned_TRADES']}/processed_orders.csv"
        else:
            finetuned_trades_path = None

        main.plot_graphs(real_path, trades_path, finetuned_trades_path, cgan_path)
        print(f"✅ {stock} {day_label} plots complete")

    print(f"\n{'='*60}")
    print("All evaluations complete!")
    print(f"{'='*60}")
    print("\nGenerated plots:")
    !ls -lh evaluation/visualizations/*.pdf
else:
    print("⏭️  Full evaluation skipped (set RUN_FULL_EVAL=True to enable)")

Running full main.py evaluation...

Generating plots for INTC Day 1 (2015-01-29)
Coverage percentage: 17.251315821970067
Coverage percentage: 36.5900120118575
Coverage percentage: 28.996863773333605
✅ INTC Day 1 (2015-01-29) plots complete

Generating plots for INTC Day 2 (2015-01-30)
Coverage percentage: 59.73637830119581
Coverage percentage: 47.17565537178628
Coverage percentage: 59.554885585953464
✅ INTC Day 2 (2015-01-30) plots complete

Generating plots for TSLA Day 1 (2015-01-29)
Coverage percentage: 37.55009289166956
Coverage percentage: 30.58532257050368
Coverage percentage: 48.614673340606416
✅ TSLA Day 1 (2015-01-29) plots complete

Generating plots for TSLA Day 2 (2015-01-30)
Coverage percentage: 56.96692600630017
Coverage percentage: 26.827346500679493
Coverage percentage: 21.093201747482134
✅ TSLA Day 2 (2015-01-30) plots complete

All evaluations complete!

Generated plots:
ls: cannot access 'evaluation/visualizations/*.pdf': No such file or directory


## 6.2. Download Results

In [ ]:
from google.colab import files

# Zip all logs
!zip -r all_simulation_results.zip ABIDES/log/ 2>/dev/null

import os
file_size = os.path.getsize('all_simulation_results.zip') / (1024 * 1024)
print(f"\n📦 Results packaged: {file_size:.1f} MB")
print("\nDownloading...")

files.download('all_simulation_results.zip')

print("✅ Download complete!")
print("\nContents:")
print("  - All simulation logs (processed_orders.csv + plots)")
print("  - Evaluation plots (if generated)")
print("  - Results summary")


  adding: ABIDES/log/ (stored 0%)
  adding: ABIDES/log/world_agent_TSLA_2015-01-30_12-00-00_30_DDIM_0.0_10_val_ema=0.811/ (stored 0%)
  adding: ABIDES/log/world_agent_TSLA_2015-01-30_12-00-00_30_DDIM_0.0_10_val_ema=0.811/corr_coef_lag_join_4way.pdf (deflated 33%)
  adding: ABIDES/log/world_agent_TSLA_2015-01-30_12-00-00_30_DDIM_0.0_10_val_ema=0.811/PCA_plot.pdf (deflated 1%)
  adding: ABIDES/log/world_agent_TSLA_2015-01-30_12-00-00_30_DDIM_0.0_10_val_ema=0.811/interarrival_time_plot_4way.pdf (deflated 27%)
  adding: ABIDES/log/world_agent_TSLA_2015-01-30_12-00-00_30_DDIM_0.0_10_val_ema=0.811/log_return_join.pdf (deflated 19%)
  adding: ABIDES/log/world_agent_TSLA_2015-01-30_12-00-00_30_DDIM_0.0_10_val_ema=0.811/order_type_join_4way.pdf (deflated 33%)
  adding: ABIDES/log/world_agent_TSLA_2015-01-30_12-00-00_30_DDIM_0.0_10_val_ema=0.811/summary_log.bz2 (stored 0%)
  adding: ABIDES/log/world_agent_TSLA_2015-01-30_12-00-00_30_DDIM_0.0_10_val_ema=0.811/world_agent_sim.png (deflated 10%)
  

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Download complete!

Contents:
  - All simulation logs (processed_orders.csv + plots)
  - Evaluation plots (if generated)
  - Results summary


---
# Summary

🎉 **Reproduction Complete!**

You have successfully reproduced the TRADES paper results.

## What was reproduced:
- ✅ Predictive scores averaged over 2 days
- ✅ TRADES vs Market Replay comparison
- ✅ Optional: CGAN baselines
- ✅ Optional: PCA, stylized facts, and other figures

## Next steps:
1. Compare your results to Table 1 in the paper
2. Examine the generated plots
3. Try different time windows or stocks
4. Modify the configuration to explore parameter sensitivity

## Citations:
If you use this reproduction in your work, please cite:
- TRADES paper: [citation]
- D-MEADS repository: https://github.com/FinancialComputingUCL/D-MEADS
